# GlitchGAN Paper — Figure Refinement

Produces publication-quality figure PDFs matching the paper style (revtex4-2 / PRD).
Figures saved to `figures/`.

**Sections**
1. Configuration & style
2. Figure 2 — DeepExtractor confusion matrix


## 1. Configuration & style

In [2]:
import io
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from IPython.display import Image, display as ipy_display
import scienceplots

# scienceplots 'science' style: Computer Modern font via usetex=True,
# matching revtex4-2 (PRD). Requires a LaTeX installation.
plt.style.use(["science"])

FIGURES_DIR = Path("figures")
FIGURES_DIR.mkdir(exist_ok=True)


def savefig(fig, name):
    """Save PDF to figures/ and display inline via BytesIO."""
    path = FIGURES_DIR / f"{name}.pdf"
    fig.savefig(path, bbox_inches="tight")
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    buf.seek(0)
    ipy_display(Image(buf.read()))
    print(f"Saved: {path}")


def _tex(label):
    """Escape underscores for LaTeX rendering."""
    return label.replace("_", r"\_")


print("Style loaded.")

Style loaded.


## 2. Figure 2 — DeepExtractor confusion matrix

Values extracted from `figures/deepextractor_confusion_matrix.png`.
7 true classes (rows) × 16 predicted classes (columns), 100 samples per class.
Style matches Figure 4 (GlitchGAN CM).

In [ ]:
TRUE_LABELS = [
    "Blip", "Fast_Scattering", "Koi_Fish", "Low_Frequency_Burst",
    "Scattered_Light", "Tomte", "Whistle",
]

PRED_LABELS = [
    "Air_Compressor", "Blip", "Blip_Low_Frequency", "Extremely_Loud",
    "Fast_Scattering", "Koi_Fish", "Light_Modulation", "Low_Frequency_Burst",
    "Low_Frequency_Lines", "No_Glitch", "Paired_Doves", "Power_Line",
    "Repeating_Blips", "Scattered_Light", "Tomte", "Whistle",
]

# Rows = true label, columns = predicted label (100 samples per class)
CM_RAW = np.array([
    #AC   Bl   BLF  EL   FS   KF   LM   LFB  LFL  NG   PD   PL   RB   SL   To   Wh
    [ 0, 100,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0],  # Blip
    [ 1,   0,   1,   0,  77,   1,   0,   2,   0,   2,   1,   1,   3,   7,   4,   0],  # Fast_Scattering
    [ 0,   2,   0,  14,   0,  79,   1,   0,   0,   0,   0,   0,   4,   0,   0,   0],  # Koi_Fish
    [ 0,   0,   0,   0,   0,   0,   0, 100,   0,   0,   0,   0,   0,   0,   0,   0],  # Low_Frequency_Burst
    [ 0,   0,   0,   0,   3,   0,   0,   2,   5,   7,   0,   0,   0,  83,   0,   0],  # Scattered_Light
    [ 0,   0,   2,   0,   0,   2,   0,   0,   0,   0,   0,   0,   0,   0,  96,   0],  # Tomte
    [ 0,   0,   0,   0,   0,   0,   0,   0,   8,   0,   0,   0,   0,   0,   0,  92],  # Whistle
], dtype=float)

assert np.all(CM_RAW.sum(axis=1) == 100), "Row sums must equal 100"

# Accuracy: look up each true label's own predicted column
# (np.trace on a non-square matrix picks the wrong diagonal when
#  columns are not aligned with true labels)
correct = sum(
    CM_RAW[i, PRED_LABELS.index(TRUE_LABELS[i])]
    for i in range(len(TRUE_LABELS))
)
accuracy = correct / CM_RAW.sum()
print(f"Correct: {int(correct)}/{int(CM_RAW.sum())}")
print(f"Accuracy: {accuracy:.2%}")

In [ ]:
# Column order: true labels first (in GAN/cDVGAN order), then remaining predicted classes
others = [l for l in PRED_LABELS if l not in TRUE_LABELS]
ordered_pred = TRUE_LABELS + others
col_idx = [PRED_LABELS.index(l) for l in ordered_pred]
cm_ordered = CM_RAW[:, col_idx]

# Drop all-zero predicted columns
nonzero_cols = cm_ordered.sum(axis=0) > 0
cm_plot   = cm_ordered[:, nonzero_cols]
pred_plot = [l for l, keep in zip(ordered_pred, nonzero_cols) if keep]

# Integer annotations — zeros shown as "0", matching Figure 4
annot = cm_plot.astype(int).astype(str)

fig_w = max(7.2, len(pred_plot) * 0.72)   # ~7.2 in = PRD double-column width
fig, ax = plt.subplots(figsize=(fig_w, 4.5))

sns.heatmap(
    cm_plot,
    annot=annot,
    fmt="",
    cmap="Blues",
    vmin=0,
    vmax=cm_plot.max(),
    linewidths=0.5,
    linecolor="0.85",
    cbar=True,
    annot_kws={"size": 7},
    xticklabels=[_tex(l) for l in pred_plot],
    yticklabels=[_tex(l) for l in TRUE_LABELS],
    ax=ax,
)

ax.set_xlabel("Predicted Label")
ax.set_ylabel("True Label")
ax.tick_params(axis="x", rotation=45)
ax.tick_params(axis="y", rotation=0)
fig.tight_layout()

savefig(fig, "deepextractor_confusion_matrix_refined")

## 3. Figure 4 — GlitchGAN confusion matrix

Values extracted from `figures/cDVGAN_CM.pdf`.
7 true classes (rows) × 11 predicted classes (columns), 25 samples per class.

In [ ]:
GAN_PRED_LABELS = [
    "Blip", "Fast_Scattering", "Koi_Fish", "Low_Frequency_Burst",
    "Scattered_Light", "Tomte", "Whistle",
    "1400Ripples", "Blip_Low_Frequency", "No_Glitch", "Violin_Mode",
]

# Rows = true label, columns = predicted label (25 samples per class)
GAN_CM_RAW = np.array([
    #Bl   FS   KF   LFB  SL   To   Wh   1400 BLF  NG   VM
    [23,   0,   0,   0,   0,   0,   2,   0,   0,   0,   0],  # Blip
    [ 0,  23,   0,   0,   2,   0,   0,   0,   0,   0,   0],  # Fast_Scattering
    [13,   0,   0,   0,   0,   2,   1,   0,   9,   0,   0],  # Koi_Fish
    [ 0,   0,   0,  25,   0,   0,   0,   0,   0,   0,   0],  # Low_Frequency_Burst
    [ 0,  13,   0,   0,  12,   0,   0,   0,   0,   0,   0],  # Scattered_Light
    [ 0,   0,   0,   1,   0,  23,   0,   0,   1,   0,   0],  # Tomte
    [ 0,   0,   0,   0,   0,   0,  10,   2,   0,  10,   3],  # Whistle
], dtype=float)

assert np.all(GAN_CM_RAW.sum(axis=1) == 25), "Row sums must equal 25"

correct_gan = sum(
    GAN_CM_RAW[i, GAN_PRED_LABELS.index(TRUE_LABELS[i])]
    for i in range(len(TRUE_LABELS))
)
accuracy_gan = correct_gan / GAN_CM_RAW.sum()
print(f"Correct: {int(correct_gan)}/{int(GAN_CM_RAW.sum())}")
print(f"Accuracy: {accuracy_gan:.2%}")

In [ ]:
# Column order: true labels first (GAN order), then remaining predicted classes
others_gan = [l for l in GAN_PRED_LABELS if l not in TRUE_LABELS]
ordered_pred_gan = TRUE_LABELS + others_gan
col_idx_gan = [GAN_PRED_LABELS.index(l) for l in ordered_pred_gan]
cm_ordered_gan = GAN_CM_RAW[:, col_idx_gan]

# Drop all-zero predicted columns
nonzero_cols_gan = cm_ordered_gan.sum(axis=0) > 0
cm_plot_gan   = cm_ordered_gan[:, nonzero_cols_gan]
pred_plot_gan = [l for l, keep in zip(ordered_pred_gan, nonzero_cols_gan) if keep]

annot_gan = cm_plot_gan.astype(int).astype(str)

fig_w = max(7.2, len(pred_plot_gan) * 0.72)
fig, ax = plt.subplots(figsize=(fig_w, 4.5))

sns.heatmap(
    cm_plot_gan,
    annot=annot_gan,
    fmt="",
    cmap="Blues",
    vmin=0,
    vmax=cm_plot_gan.max(),
    linewidths=0.5,
    linecolor="0.85",
    cbar=True,
    annot_kws={"size": 7},
    xticklabels=[_tex(l) for l in pred_plot_gan],
    yticklabels=[_tex(l) for l in TRUE_LABELS],
    ax=ax,
)

ax.set_xlabel("Predicted Label")
ax.set_ylabel("True Label")
ax.tick_params(axis="x", rotation=45)
ax.tick_params(axis="y", rotation=0)
fig.tight_layout()

savefig(fig, "glitchgan_confusion_matrix_refined")